# SMART Disk Failure Classifier
Random Forest Classifier pipeline — data preparation + training.

## 1. IMPORTS

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib

## 2. LOAD DATA

In [ ]:
df = pd.read_csv('koncniPodatkiZaModel.csv', sep=',')
display(df.head(5))
display(df.shape)

## 3. ROUGH PREPROCESSING — Column Selection

In [ ]:
nan_ratio = df.isnull().mean()

required_columns = [
    'smart_5_raw',
    'smart_187_raw',
    'smart_188_raw',
    'smart_197_raw',
    'smart_198_raw',
    'smart_190_raw',
    'smart_191_raw',
    'smart_9_raw',
    'smart_194_raw',
    'capacity_bytes',
    'model',
    'failure'
]

# Keep column if < 30% NaN OR it is in the required list
relevant_cols = nan_ratio[(nan_ratio < 0.3) | (nan_ratio.index.isin(required_columns))].index.tolist()
cols_to_drop = [col for col in df.columns if col not in relevant_cols]
df = df.drop(columns=cols_to_drop)

# Drop location columns (Backblaze physical server metadata — irrelevant to disk health)
location_cols = ['datacenter', 'cluster_id', 'vault_id', 'pod_id', 'is_legacy_format']
df = df.drop(columns=[c for c in location_cols if c in df.columns])

# Drop identifier columns
identifier_cols = ['date', 'serial_number']
df = df.drop(columns=[c for c in identifier_cols if c in df.columns])

# Drop all 'normalized' columns — manufacturer-specific, inconsistent across 57 brands
normalized_cols = [col for col in df.columns if 'normalized' in col]
df = df.drop(columns=normalized_cols)

print(f'Remaining columns: {df.shape[1]}')
display(df.columns)

## 4. ENCODE DISK TYPE & MANUFACTURER

In [ ]:
# SSD flag — 1 if solid-state, 0 if mechanical
ssd_keywords = ['SSD', 'MTFD', 'SSDSC', '850 PRO', '870 EVO', '860 PRO', '5300']
df['is_ssd'] = df['model'].apply(lambda x: 1 if any(k in str(x).upper() for k in ssd_keywords) else 0)

# Normalize raw model strings into manufacturer groups
def get_manufacturer(model):
    m = str(model).upper()
    if m.startswith('ST') or 'SEAGATE' in m: return 'Seagate'
    if m.startswith('WDC') or m.startswith('WD') or 'WESTERN' in m: return 'Western Digital'
    if m.startswith('HGST') or m.startswith('HUH') or m.startswith('HMS'): return 'HGST'
    if m.startswith('TOSHIBA') or m.startswith('MG'): return 'Toshiba'
    if m.startswith('SAMSUNG'): return 'Samsung'
    if m.startswith('CT') or 'CRUCIAL' in m: return 'Crucial'
    return 'Other'

df['model'] = df['model'].apply(get_manufacturer)
display(df['model'].value_counts())

## 5. NaN IMPUTATION (Domain-Aware)

In [ ]:
# Convert capacity bytes -> gigabytes
df = df.rename(columns={'capacity_bytes': 'capacity_gigabytes'})
df['capacity_gigabytes'] = (df['capacity_gigabytes'] / (1024 ** 3)).round(2)

# Error counters: missing = no errors recorded -> fill with 0
error_counter_cols = ['smart_1_raw', 'smart_7_raw', 'smart_10_raw', 'smart_192_raw', 'smart_199_raw', 'smart_191_raw']
df[[c for c in error_counter_cols if c in df.columns]] = df[[c for c in error_counter_cols if c in df.columns]].fillna(0)

# Temperature: fill with median (missing sensor report, not actually 0 degrees)
if 'smart_190_raw' in df.columns:
    df['smart_190_raw'] = df['smart_190_raw'].fillna(df['smart_190_raw'].median())

# Mechanical cycle columns: SSDs get 0 (no moving parts), HDDs get median
mechanical_cols = ['smart_3_raw', 'smart_4_raw', 'smart_193_raw']
for col in [c for c in mechanical_cols if c in df.columns]:
    df.loc[(df['is_ssd'] == 1) & (df[col].isnull()), col] = 0
    hdd_median = df.loc[df['is_ssd'] == 0, col].median()
    df.loc[(df['is_ssd'] == 0) & (df[col].isnull()), col] = hdd_median

# Power cycles: every disk powers on, 0 would be unrealistic -> use median
if 'smart_12_raw' in df.columns:
    df['smart_12_raw'] = df['smart_12_raw'].fillna(df['smart_12_raw'].median())

# Critical SMART flags: missing = no detected problem -> fill with 0
critical_required = ['smart_5_raw', 'smart_9_raw', 'smart_187_raw', 'smart_188_raw', 'smart_194_raw', 'smart_197_raw', 'smart_198_raw']
df[[c for c in critical_required if c in df.columns]] = df[[c for c in critical_required if c in df.columns]].fillna(0)

# Drop low-correlation columns identified during feature selection
low_signal_cols = ['smart_190_raw', 'smart_194_raw', 'smart_199_raw', 'smart_10_raw']
df.drop(columns=[c for c in low_signal_cols if c in df.columns], inplace=True)

print('NaN remaining:', df.isnull().sum().sum())
display(df.head())

## 6. FEATURE ENGINEERING

In [ ]:
critical_params = [c for c in ['smart_5_raw', 'smart_187_raw', 'smart_197_raw', 'smart_198_raw'] if c in df.columns]

# Binary flag: did ANY critical SMART error occur?
df['any_critical_error'] = df[critical_params].sum(axis=1).apply(lambda x: 1 if x > 0 else 0)

# Total count of all critical errors combined
df['total_error_count'] = df[critical_params].sum(axis=1)

# Error rate normalized by disk size
df['error_per_gb'] = df['total_error_count'] / df['capacity_gigabytes']

print(df[['any_critical_error', 'total_error_count', 'error_per_gb', 'failure']].head(10))

## 7. PREPARE FEATURES AND TARGET

In [ ]:
X = df.drop(columns=['model', 'failure'])
y = df['failure']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Training set: {X_train.shape}')
print(f'Test set:     {X_test.shape}')

## 8. BASELINE RANDOM FOREST CLASSIFIER

In [ ]:
baseline_rf = RandomForestClassifier(n_estimators=100, random_state=42)
baseline_rf.fit(X_train, y_train)
y_pred_baseline = baseline_rf.predict(X_test)

print(classification_report(y_test, y_pred_baseline))
print(f'Accuracy: {accuracy_score(y_test, y_pred_baseline)*100:.2f}%')

cm_baseline = confusion_matrix(y_test, y_pred_baseline)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_baseline, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Healthy (0)', 'Failed (1)'],
            yticklabels=['Healthy (0)', 'Failed (1)'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix — Baseline Random Forest')
plt.show()

## 9. CROSS-VALIDATION (stability check)

In [ ]:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(baseline_rf, X, y, cv=kfold, scoring='accuracy')

print(f'Scores per fold: {cv_scores}')
print(f'Mean accuracy:   {np.mean(cv_scores)*100:.2f}%')
print(f'Std deviation:   {np.std(cv_scores)*100:.2f}%')

## 10. GRIDSEARCHCV — Hyperparameter Optimization

In [ ]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'class_weight': ['balanced', None]
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=3,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)
print(f'Best parameters: {grid_search.best_params_}')

## 11. EVALUATE OPTIMIZED MODEL

In [ ]:
best_model = grid_search.best_estimator_
y_pred_optimized = best_model.predict(X_test)

print('--- Results after GridSearchCV ---')
print(classification_report(y_test, y_pred_optimized))
print(f'Accuracy: {accuracy_score(y_test, y_pred_optimized)*100:.2f}%')

cm_optimized = confusion_matrix(y_test, y_pred_optimized)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_optimized, annot=True, fmt='d', cmap='Greens', cbar=False,
            xticklabels=['Healthy (0)', 'Failed (1)'],
            yticklabels=['Healthy (0)', 'Failed (1)'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix — Optimized Random Forest')
plt.show()

## 12. FEATURE IMPORTANCE

In [ ]:
importances = best_model.feature_importances_
feature_importance_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances})
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df,
            hue='Feature', palette='magma', legend=False)
plt.title('Feature Importance — Optimized Random Forest')
plt.xlabel('Importance (influence on model decision)')
plt.ylabel('SMART Parameter')
plt.show()

## 13. SAVE MODEL

In [ ]:
joblib.dump(best_model, 'disk_model.pkl')
feature_importance_df.to_csv('feature_importance.csv', index=False)
print("Saved: 'disk_model.pkl' and 'feature_importance.csv'")